In [1]:
import os
os.chdir("../")

In [2]:
%pwd


'c:\\Users\\user\\iscale\\Kidney--Disease--Classification'

In [68]:
from dataclasses import dataclass
from pathlib import Path

@dataclass(frozen=True)
class TrainingConfig:
    root_dir: Path
    base_model_path: Path
    trained_model_path: Path
    updated_base_model_path: Path
    training_data: Path
    params_epochs: int
    params_batch_size: int
    params_is_augmentation: bool
    params_image_size: list

In [69]:
from Kidney.constants import *
from Kidney.utils.common import read_yaml ,create_directories
import tensorflow as tf



In [70]:
class ConfigurationManager:
    def __init__(
            self,
            config_filepath=CONFIG_FILE_PATH,
            params_filepath=PARAMS_FILE_PATH):
            
            self.config=read_yaml(config_filepath)
            self.params=read_yaml(params_filepath)

            create_directories([self.config.artifacts_root])
    def get_training_config(self)->TrainingConfig:
        training=self.config.training
        prepare_base_model=self.config.prepare_base_model
        params=self.params
        training_data=os.path.join(self.config.data_ingestion.unzip_dir,"CT-KIDNEY-DATASET-Normal-Cyst-Tumor-Stone")

        create_directories([Path(training.root_dir)])
        training_config = TrainingConfig(
        root_dir=Path(training.root_dir),
        training_data=Path(training_data),
        trained_model_path=Path(training.trained_model_path),
        base_model_path=Path(prepare_base_model.base_model_path),
        updated_base_model_path=Path(prepare_base_model.updated_base_model_path),
        params_epochs=self.params.EPOCHS,
        params_batch_size=self.params.BATCH_SIZE,
        params_is_augmentation=self.params.AUGMENTATION,
        params_image_size=self.params.IMAGE_SIZE
        )

    
        
          
        return training_config
    

In [71]:
import os 
import urllib.request as request
from zipfile import ZipFile
import tensorflow as tf
import time


In [72]:
class Training:
    def __init__(self,config: TrainingConfig):
        self.config=config
    
    def get_base_model(self):
        
        self.model=tf.keras.models.load_model(
                self.config.updated_base_model_path
            )
    def train_valid_generator(self):
        datagenerator_kwargs=dict(
            rescale=1./255,
            validation_split=0.2
        )
        dataflow_kwargs=dict(
            target_size=self.config.params_image_size[:-1],
            batch_size=self.config.params_batch_size,
            interpolation="bilinear"
        )
        valid_datagenerator=tf.keras.preprocessing.image.ImageDataGenerator(
            **datagenerator_kwargs)
        self.valid_generator=valid_datagenerator.flow_from_directory(
            directory=self.config.training_data,
            subset="validation",
            shuffle=False,
            **dataflow_kwargs)

        
        if self.config.params_is_augmentation:
            train_datagenerator=tf.keras.preprocessing.image.ImageDataGenerator(
                rotation_range=40,
                horizontal_flip=True,
                width_shift_range=0.2,
                height_shift_range=0.2,
                shear_range=0.2,
                zoom_range=0.2,
                **datagenerator_kwargs
            )
        else:
            train_datagenerator=valid_datagenerator
        self.train_generator=train_datagenerator.flow_from_directory(
            directory=self.config.training_data,
            subset="training",
            shuffle=True,
            **dataflow_kwargs
        )



    @staticmethod
    def save_model(path:Path,model :tf.keras.Model):
        model.save(path)



    def train(self,callback_list: list):
        
        self.steps_per_epoch=self.train_generator.samples // self.train_generator.batch_size
        self.validation_steps=self.valid_generator.samples // self.valid_generator.batch_size

        self.model.fit(
            self.train_generator,
            epochs=self.config.params_epochs,
            steps_per_epoch=self.steps_per_epoch,
            validation_steps=self.validation_steps,
            validation_data=self.valid_generator,
            callbacks=callback_list

        )
        self.save_model(
            path=self.config.trained_model_path,
            model=self.model
        )

In [73]:
try:
    config=ConfigurationManager()
    training_config= config.get_training_config()
    training=Training(config=training_config)
    training.get_base_model()
    training.train_valid_generator()
    training.train(callback_list=[]
        
)
except Exception as e:
    raise e

[2026-06-12 12:32:02,641: INFO: common: yaml file : config\config.yaml loaded successfully ]
[2026-06-12 12:32:02,695: INFO: common: yaml file : PARAMS.yaml loaded successfully ]
[2026-06-12 12:32:02,708: INFO: common: created  directory at : artifacts]
[2026-06-12 12:32:02,720: INFO: common: created  directory at : artifacts\training]
Found 2489 images belonging to 1 classes.
Found 9957 images belonging to 1 classes.
 17/311 [>.............................] - ETA: 1:11:17 - loss: 32.5278 - accuracy: 0.5239

UnknownError: Graph execution error:

FileNotFoundError: [Errno 2] No such file or directory: 'artifacts\\data_ingestion\\CT-KIDNEY-DATASET-Normal-Cyst-Tumor-Stone\\CT-KIDNEY-DATASET-Normal-Cyst-Tumor-Stone\\Normal\\Normal- (689).jpg'
Traceback (most recent call last):

  File "c:\Users\user\anaconda3\New folder\envs\kidney\lib\site-packages\tensorflow\python\ops\script_ops.py", line 267, in __call__
    ret = func(*args)

  File "c:\Users\user\anaconda3\New folder\envs\kidney\lib\site-packages\tensorflow\python\autograph\impl\api.py", line 642, in wrapper
    return func(*args, **kwargs)

  File "c:\Users\user\anaconda3\New folder\envs\kidney\lib\site-packages\tensorflow\python\data\ops\from_generator_op.py", line 198, in generator_py_func
    values = next(generator_state.get_iterator(iterator_id))

  File "c:\Users\user\anaconda3\New folder\envs\kidney\lib\site-packages\keras\engine\data_adapter.py", line 902, in wrapped_generator
    for data in generator_fn():

  File "c:\Users\us